In [1]:
import sys
print(sys.executable)

/opt/venv/bin/python3


In [2]:
import pandas
import openpyxl
print("Both packages are available")


Both packages are available


In [3]:
%pip install openpyxl


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
%run ./convert_ercot.py

PASS: 6,551 rows; no missing values, duplicates, or hourly gaps.
Saved: data/ercot_hourly.csv
Quality report: data/ercot_hourly.quality.json


In [7]:
%run scripts/train_demand.py data/ercot_hourly.csv --out artifacts/demand

{
  "source_csv": "/home/ibm-user/shared/AllPower/data/ercot_hourly.csv",
  "rows": 6551,
  "model": "XGBoost",
  "selected_on_validation": {
    "1": "candidate",
    "2": "candidate",
    "3": "candidate",
    "4": "candidate",
    "5": "candidate",
    "6": "weekly_baseline"
  },
  "split": {
    "train_end": "2026-07-10T21:00:00+00:00",
    "validation_start": "2026-07-11T04:00:00+00:00",
    "test_start": "2026-08-21T02:00:00+00:00"
  },
  "horizons": {
    "1": {
      "validation_candidate": {
        "mae_mw": 2092.3798828125,
        "rmse_mw": 2959.7
      },
      "validation_weekly_baseline": {
        "mae_mw": 3476.46,
        "rmse_mw": 5365.12
      },
      "candidate": {
        "mae_mw": 2213.080078125,
        "rmse_mw": 2982.97
      },
      "weekly_baseline": {
        "mae_mw": 2301.45,
        "rmse_mw": 2855.29
      },
      "persistence_baseline": {
        "mae_mw": 2347.96,
        "rmse_mw": 2741.67
      },
      "validation_interval_half_width_mw": 5370

In [8]:
import platform
from pathlib import Path

print("Machine architecture:", platform.machine())
print("Working folder:", Path.cwd())

for file in sorted(Path("artifacts/demand").glob("*")):
    print(file.name, file.stat().st_size, "bytes")

Machine architecture: s390x
Working folder: /home/ibm-user/shared/AllPower
.ipynb_checkpoints 4096 bytes
evaluation.json 4028 bytes
h1.pkl 314706 bytes
h2.pkl 315318 bytes
h3.pkl 315522 bytes
h4.pkl 316270 bytes
h5.pkl 316336 bytes
h6.pkl 317968 bytes


In [9]:
import importlib

for name in ["numpy", "PIL", "torch", "sklearn", "skimage"]:
    try:
        module = importlib.import_module(name)
        print(name, "OK", getattr(module, "__version__", ""))
    except Exception as error:
        print(name, "FAILED:", error)

numpy OK 2.0.2
PIL OK 12.3.0
torch OK 2.11.0+cpu
sklearn OK 1.9.0
skimage OK 0.26.0


In [10]:
from pathlib import Path
import zipfile

assert Path("scripts/train_vision.py").exists(), "Open a notebook in the AllPower folder."

with zipfile.ZipFile("AllPower_Vision_Demo.zip") as archive:
    existing = [
        name for name in archive.namelist()
        if Path(name).exists()
    ]
    if existing:
        raise RuntimeError("Some package files already exist. Tell me before continuing.")
    archive.extractall(".")

print("Image dataset and training script are ready.")

Image dataset and training script are ready.


In [11]:
%run train_vision_demo.py data/vision_demo --out artifacts/vision

Training HOG baseline first...
HOG model saved. Training CNN for at most 5 epochs...
Epoch 1/5: validation balanced accuracy 0.500
Epoch 2/5: validation balanced accuracy 0.500
Epoch 3/5: validation balanced accuracy 0.500
{
  "dataset": {
    "dataset": "InsPLAD glass-insulator demo subset",
    "source_url": "https://data.mendeley.com/datasets/5n3fjgvfyz/1",
    "license": "CC BY-NC 3.0",
    "attribution": "Vieira-e-Silva et al., InsPLAD, 2023, DOI 10.17632/5n3fjgvfyz.1",
    "counts": {
      "train": {
        "normal": 547,
        "defect": 50
      },
      "val": {
        "normal": 143,
        "defect": 9
      },
      "test": {
        "normal": 29,
        "defect": 30
      }
    },
    "augmented_files_excluded": 621,
    "original_training_records_excluded_for_test_overlap": 11,
    "seed": 19,
    "split_policy": "Publisher val reserved as test; remove training source-photo/pixel overlaps with test; split remaining train by source-photo ID into train/val; remove pixel